# Clustering Algorithms: Theory, Math, and Implementation

Clustering is an **unsupervised machine learning** technique used to group identical or similar data points into clusters. Unlike supervised learning (classification/regression), the algorithm is not given labeled targets ($y$); instead, it must discover the hidden structures and patterns within the input features ($X$) on its own.

In this notebook, we will explore three fundamental clustering algorithms, understand their mathematical foundations, and visualize how they group data using Python and `scikit-learn`.

---
### Setup & Imports
We will generate two different types of datasets:
1. **Spherical Blobs:** Good for testing centroid-based models like K-Means.
2. **Interlocking Moons:** Non-linear, arbitrarily shaped data that is perfect for testing density-based models like DBSCAN.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, FloatLogSlider

plt.style.use('seaborn-v0_8-darkgrid')

# Generate Spherical Data (For K-Means & Hierarchical)
np.random.seed(42)
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

# Generate Arbitrary Non-linear Data (For DBSCAN)
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=0)


---

### 1. K-Means Clustering

K-Means is a centroid-based clustering algorithm. It partitions the dataset into exactly $K$ non-overlapping clusters. The algorithm randomly initializes $K$ center points (centroids) and iteratively assigns each data point to the closest centroid, then recalculates the centroid's position based on the mean of the assigned points.

**Mathematical Foundation:**
The objective of K-Means is to minimize the **Within-Cluster Sum of Squares (WCSS)**, also known as Inertia. This represents the total squared distance between each data point ($x_i$) and its assigned cluster centroid ($c_j$).

$$J = \sum_{j=1}^{K} \sum_{i=1}^{n_j} ||x_i^{(j)} - c_j||^2$$

Where:
* $K$ is the total number of clusters.
* $n_j$ is the number of data points in the $j$-th cluster.
* $x_i^{(j)}$ is a data point belonging to cluster $j$.
* $c_j$ is the centroid of cluster $j$.
* $||x - c||^2$ is the squared Euclidean distance.

**Example Problem:**
* **Marketing:** Customer segmentation—grouping customers into "Budget", "Spenders", and "Frequent Buyers" based on numerical purchase history to target specific advertisements.

Use the slider below to adjust $K$. Notice what happens if $K$ is too small (under-segmentation) or too large (over-segmentation splitting natural groups).

In [ ]:
@interact(n_clusters=IntSlider(min=1, max=10, step=1, value=4, description='K (Clusters)'))
def plot_kmeans(n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    y_kmeans = kmeans.fit_predict(X_blobs)
    centers = kmeans.cluster_centers_

    plt.figure(figsize=(8, 5))
    plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_kmeans, s=50, cmap='viridis', edgecolor='k', alpha=0.7)
    plt.scatter(centers[:, 0], centers[:, 1], c='red', s=250, marker='X', edgecolor='white', label='Centroids')
    plt.title(f"K-Means Clustering (K={n_clusters}) | Inertia: {kmeans.inertia_:.2f}", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.show()

interactive(children=(IntSlider(value=4, description='K (Clusters)', max=10, min=1), Output()), _dom_classes=(…

### 2. Hierarchical Clustering (Agglomerative)

Unlike K-Means, which requires you to guess $K$ upfront, Hierarchical clustering builds a hierarchy of clusters. 

**Agglomerative** is a "bottom-up" approach: 
1. It starts by treating every single data point as its own individual cluster. 
2. It calculates the distance between all pairs of clusters.
3. It repeatedly merges the two "closest" clusters into a larger cluster.
4. This repeats until only one massive cluster (containing all data points) remains. 

This nested, tree-like relationship can be beautifully visualized as a **Dendrogram**. By cutting the Dendrogram at a specific height, you can choose exactly how many clusters you want.

**Mathematical Foundation: The Linkage Criteria**

Before we can merge clusters, we need a base distance metric between individual points, usually the Euclidean distance: $d(x, y) = ||x - y||_2$. 

However, how do we measure the distance between *groups* (clusters) of points? If we have Cluster $A$ and Cluster $B$, the "distance" between them depends on the **Linkage Criterion** we choose:

**1. Ward's Linkage (Default in Scikit-Learn):**
Instead of just looking at the distance between points, Ward's method looks at **Variance**. It merges the two clusters that will result in the smallest *increase* in the total Within-Cluster Sum of Squares (WCSS).
Mathematically, the distance (increase in variance) when merging Cluster $A$ and Cluster $B$ is:
$$\Delta(A, B) = \frac{|A| \times |B|}{|A| + |B|} ||\mu_A - \mu_B||^2$$
Where:
* $|A|$ and $|B|$ are the number of points in clusters $A$ and $B$.
* $\mu_A$ and $\mu_B$ are the centroids (means) of clusters $A$ and $B$.
* *Result:* Creates highly spherical, evenly sized clusters (very similar to K-Means).

**2. Complete Linkage (Maximum Distance):**
This criterion considers the distance between two clusters to be the maximum distance between their two *furthest* points.
$$D(A, B) = \max_{x \in A, y \in B} d(x, y)$$
* *Result:* Forces clusters to be tightly bound and compact, avoiding long, sprawling clusters.

**3. Average Linkage:**
This calculates the distance between all possible pairs of points (one from Cluster A, one from Cluster B) and takes the average.
$$D(A, B) = \frac{1}{|A| \times |B|} \sum_{x \in A} \sum_{y \in B} d(x, y)$$
* *Result:* A great middle-ground that is robust to outliers and creates distinct, cohesive groups.

**4. Single Linkage (Minimum Distance):**
This considers the distance between two clusters to be the minimum distance between their two *closest* points.
$$D(A, B) = \min_{x \in A, y \in B} d(x, y)$$
* *Result:* Susceptible to the **"Chaining Effect"**. It will easily merge two massive clusters if just one point from each cluster happens to be close together, often resulting in long, string-like clusters.

**Example Problem:**
* **Genetics & Biology:** Grouping animal species based on genetic similarity to reconstruct an evolutionary tree (phylogeny). A Dendrogram perfectly represents when species diverged from common ancestors.

Use the dropdown below to change the linkage method. Notice how `ward` forces tight, even clusters, while `single` linkage can sometimes create long, stringy chains across the data space.

In [ ]:
@interact(n_clusters=IntSlider(min=2, max=10, step=1, value=4, description='Clusters'),
          linkage_method=Dropdown(options=['ward', 'complete', 'average', 'single'], value='ward', description='Linkage'))
def plot_hierarchical(n_clusters, linkage_method):
    hc = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method)
    y_hc = hc.fit_predict(X_blobs)

    plt.figure(figsize=(8, 5))
    plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_hc, s=50, cmap='plasma', edgecolor='k', alpha=0.7)
    
    plt.title(f"Hierarchical Clustering (Clusters={n_clusters}, Linkage={linkage_method.capitalize()})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

interactive(children=(IntSlider(value=4, description='Clusters', max=10, min=2), Dropdown(description='Linkage…

### 3. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

K-Means and Ward Hierarchical clustering assume that clusters are spherical (round). If the data is arranged in complex, overlapping, or arbitrary shapes (like our interlocking moons), centroid-based models fail. DBSCAN solves this by grouping together points that are closely packed together (high density), while marking points that lie alone in low-density regions as **Noise (Outliers)**.

**Mathematical Foundation:**
DBSCAN relies on two crucial parameters to define "density":
1. **$\epsilon$ (Epsilon / Radius):** The maximum distance between two points for them to be considered neighbors.
2. **MinPts (`min_samples`):** The minimum number of points required within the $\epsilon$ radius to form a dense region (a cluster).

DBSCAN classifies every point into three categories:
* **Core Point:** A point with at least `MinPts` neighbors within its $\epsilon$ radius.
* **Border Point:** A point that has fewer than `MinPts` neighbors, but is reachable from a Core point.
* **Noise (Outlier):** A point that is neither a core nor a border point.

DBSCAN groups points together using mathematical definitions of connection:
1. **Directly Density-Reachable:** Point $A$ is directly reachable from Point $B$ if $B$ is a Core point and $A$ is inside $B$'s $\epsilon$-neighborhood.
2. **Density-Reachable:** Point $A$ is reachable from Point $B$ if there is a chain of Core points connecting them (e.g., $p_1 \rightarrow p_2 \rightarrow p_3$, where $p_1=B$ and $p_3=A$).
3. **The Algorithm Step-by-Step:** * It picks a random unvisited point. 
   * If it's a Core point, it creates a new cluster and absorbs all density-reachable points into this cluster. 
   * It repeats this until all points are visited. If a point doesn't belong to any core point's neighborhood, it is marked as Noise (-1).

**Example Problem:**
* **Astronomy:** Identifying clusters of stars or galaxies in the sky while successfully ignoring the random scatter of background radiation (noise).

Use the sliders below to adjust `eps` and `min_samples`.
* **If `eps` is too small:** The search radius is tiny. Very few points will meet the `min_samples` requirement, and almost the entire dataset will be painted black (classified as Noise).
* **If `eps` is too large:** The search radius is huge. The algorithm will jump across the empty space between the two interlocking moons, merging them incorrectly into one giant cluster.
* **Finding the Sweet Spot:** You must balance `eps` and `min_samples` so the algorithm can "walk" along the dense path of each moon without jumping the gap!

In [ ]:
@interact(eps=FloatSlider(min=0.05, max=0.4, step=0.01, value=0.15, description='Epsilon (Radius)'),
          min_samples=IntSlider(min=2, max=15, step=1, value=5, description='Min Samples'))
def plot_dbscan(eps, min_samples):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    y_dbscan = dbscan.fit_predict(X_moons)

    plt.figure(figsize=(8, 5))
    
    unique_labels = set(y_dbscan)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = [0, 0, 0, 1]
            marker_size = 30
            label = 'Noise (Outliers)'
        else:
            marker_size = 70
            label = f'Cluster {k}'

        class_member_mask = (y_dbscan == k)
        
        xy = X_moons[class_member_mask]
        plt.scatter(xy[:, 0], xy[:, 1], c=[col], s=marker_size, edgecolor='k', alpha=0.8, label=label)

    plt.title(f"DBSCAN Clustering (eps={eps:.2f}, MinPts={min_samples})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    
    if len(unique_labels) < 10:
        plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1))
        
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.15, description='Epsilon (Radius)', max=0.4, min=0.05, step=0.01), I…

### 4. Gaussian Mixture Models (GMM)

Gaussian Mixture Models (GMM) are a probabilistic, unsupervised model. Instead of drawing hard boundaries like K-Means, GMM assumes that the entire dataset is generated from a mixture of a finite number ($K$) of Gaussian (Normal) distributions. 

GMM performs **Soft Clustering**. Instead of saying "Point A belongs to Cluster 1," GMM says "Point A has a 90% probability of belonging to Cluster 1, and a 10% probability of belonging to Cluster 2."

**Mathematical Foundation (Expectation-Maximization):**
GMM relies on the **Expectation-Maximization (EM) algorithm** to find the best fit:
1. **Initialization:** Randomly guess the parameters: mean ($\mu$), covariance ($\Sigma$, which determines the shape/spread), and mixing weight ($\pi$, the size of the cluster).
2. **Expectation (E-Step):** Calculate the probability that each data point belongs to each cluster, given the current parameters.
    $$\gamma(z_{nk}) = \frac{\pi_k \mathcal{N}(x_n | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_n | \mu_j, \Sigma_j)}$$
3. **Maximization (M-Step):** Update the parameters ($\mu, \Sigma, \pi$) to maximize the likelihood of the data based on the probabilities calculated in the E-step.
4. **Repeat:** Repeat E and M steps until the parameters stop changing (convergence).

**Covariance Types:**
The true power of GMM lies in its covariance matrix ($\Sigma$), which dictates the shape of the clusters:
* **Spherical:** Like K-Means, clusters must be perfectly round (variance is equal in all directions).
* **Diag:** Clusters can be ellipses, but the axes must align with the X and Y axes.
* **Full:** Clusters can be ellipses of any shape, size, and diagonal orientation!

**Example Problem:**
* **Fraud / Anomaly Detection:** A transaction that has a very low probability of belonging to *any* of the normal Gaussian profiles is flagged as a highly probable anomaly.

In [ ]:
@interact(
    n_components=IntSlider(min=1, max=6, step=1, value=4, description='K Clusters'),
    cov_type=Dropdown(options=['spherical', 'diag', 'tied', 'full'], value='full', description='Covariance'),
    stretch_data=FloatSlider(min=0.0, max=2.0, step=0.5, value=0.0, description='Stretch Data')
)
def plot_advanced_gmm(n_components, cov_type, stretch_data):
    X_mod = X_blobs.copy()
    if stretch_data > 0:
        transformation = np.array([[1.0, stretch_data], [0.0, 1.0]])
        X_mod = np.dot(X_mod, transformation)
        
    gmm = GaussianMixture(n_components=n_components, covariance_type=cov_type, random_state=42)
    y_gmm = gmm.fit_predict(X_mod)
    
    plt.figure(figsize=(10, 6))
    
    x_min, x_max = X_mod[:, 0].min() - 1, X_mod[:, 0].max() + 1
    y_min, y_max = X_mod[:, 1].min() - 1, X_mod[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    
    Z = -gmm.score_samples(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contour(xx, yy, Z, levels=np.logspace(0, 2, 8), cmap='Reds', alpha=0.5)
    
    plt.scatter(X_mod[:, 0], X_mod[:, 1], c=y_gmm, s=40, cmap='viridis', edgecolor='k', alpha=0.7)
    

    centers = gmm.means_
    weights = gmm.weights_
    
    for i in range(n_components):
        plt.scatter(centers[i, 0], centers[i, 1], c='red', s=200, marker='X', edgecolor='white')
        plt.text(centers[i, 0], centers[i, 1] + 0.6, f"Weight: {weights[i]:.2f}", 
                 fontsize=10, weight='bold', color='black', ha='center', 
                 bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title(f"GMM (K={n_components}, Covariance: {cov_type})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.show()

interactive(children=(IntSlider(value=4, description='K Clusters', max=6, min=1), Dropdown(description='Covari…

### 5. Spectral Clustering

Spectral Clustering is a technique rooted in **Graph Theory**. Instead of looking at the data points as coordinates in space, it treats them as "nodes" in a highly connected network (a graph). It cuts the weakest "edges" (links) between the nodes to separate them into disconnected communities (clusters).

It is incredibly powerful for identifying clusters with complex, non-linear boundaries, such as concentric circles.

**Mathematical Foundation:**
1. **The Similarity Graph (Affinity Matrix $A$):** Compute a similarity score between every pair of points. Often done using a Radial Basis Function (RBF) kernel:
   $$A_{ij} = \exp\left(-\gamma ||x_i - x_j||^2\right)$$
   *(Points close together have affinity near 1; far apart have affinity near 0).*
   
2. **The Graph Laplacian ($L$):** Calculate the Degree matrix $D$ (sum of affinities for each point) and find the unnormalized Graph Laplacian:
   $$L = D - A$$

3. **Eigenvalue Decomposition (The "Spectrum"):** Find the eigenvalues and eigenvectors of the matrix $L$. The algorithm uses the lowest $K$ eigenvectors to project the data into a brand new, lower-dimensional space.
   *(In this new mathematical space, complex shapes like interlocking rings are magically "unrolled" into dense, easily separable blobs!)*

4. **Final Clustering:** Finally, apply standard K-Means clustering to this newly transformed data to get the final labels.

**Example Problem:**
* **Social Network Analysis:** Identifying tight-knit communities or friend groups within a massive network of user connections.
* **Computer Vision:** Image segmentation, where pixels are grouped into objects based on color similarity and physical proximity.

In [18]:
from sklearn.cluster import SpectralClustering
from sklearn.datasets import make_circles

X_circles, _ = make_circles(n_samples=400, factor=0.4, noise=0.05, random_state=42)

@interact(gamma=FloatLogSlider(value=10.0, min=-1, max=3, step=0.1, description='Gamma (RBF)'))
def plot_spectral(gamma):
   
    spectral = SpectralClustering(n_clusters=2, affinity='rbf', gamma=gamma, random_state=42)
    y_spectral = spectral.fit_predict(X_circles)
    
    plt.figure(figsize=(8, 5))
    plt.scatter(X_circles[:, 0], X_circles[:, 1], c=y_spectral, s=40, cmap='coolwarm', edgecolor='k')
    
    plt.title(f"Spectral Clustering on Non-Linear Data (Gamma={gamma:.1f})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

interactive(children=(FloatLogSlider(value=10.0, description='Gamma (RBF)', max=3.0, min=-1.0), Output()), _do…

In [ ]:
from sklearn.cluster import SpectralClustering
from ipywidgets import interact, FloatLogSlider

@interact(gamma=FloatLogSlider(value=20.0, min=0, max=3, step=0.5, description='Gamma (RBF)'))
def plot_spectral_moons(gamma):
    
    spectral = SpectralClustering(n_clusters=2, affinity='rbf', gamma=gamma, random_state=42)
    y_spectral = spectral.fit_predict(X_moons)
    
    plt.figure(figsize=(8, 5))
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_spectral, s=50, cmap='coolwarm', edgecolor='k', alpha=0.8)
    
    plt.title(f"Spectral Clustering on Interlocking Moons (Gamma={gamma:.1f})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

interactive(children=(FloatLogSlider(value=20.0, description='Gamma (RBF)', max=3.0, step=0.5), Output()), _do…

### 6. Affinity Propagation

Unlike K-Means or K-Medoids, Affinity Propagation does not require you to choose the number of clusters ($K$) or randomly initialize cluster centers. Instead, all data points are simultaneously considered as potential "exemplars" (cluster centers). 



The points communicate with each other by passing real-valued messages through the network until a high-quality set of exemplars and clusters gradually emerges.

**Mathematical Foundation (Message Passing):**
The algorithm relies on two types of messages passed between data points $i$ and $k$:

1. **Similarity $s(i, k)$:** How well-suited point $k$ is to be the exemplar for point $i$. Usually the negative squared Euclidean distance:
   $$s(i, k) = -||x_i - x_k||^2$$
   *(The similarity $s(k, k)$ is called the **preference**. A higher preference means point $k$ is more likely to become an exemplar, leading to more clusters).*

2. **Responsibility $r(i, k)$:** Sent from point $i$ to candidate exemplar $k$. It reflects the accumulated evidence of how well-suited $k$ is to serve as the exemplar for $i$, compared to all other candidate exemplars.
   $$r(i, k) \leftarrow s(i, k) - \max_{k' \neq k} \left\{ a(i, k') + s(i, k') \right\}$$

3. **Availability $a(i, k)$:** Sent from candidate exemplar $k$ back to point $i$. It reflects the accumulated evidence of how appropriate it would be for $i$ to choose $k$ as its exemplar, based on how many other points currently favor $k$.
   $$a(i, k) \leftarrow \min \left( 0, r(k, k) + \sum_{i' \notin \{i, k\}} \max(0, r(i', k)) \right)$$

These messages are updated iteratively. To avoid numerical oscillations, a **Damping factor** ($\lambda$ between 0.5 and 1.0) is applied to the updates:
$$msg_{new} = (\lambda \times msg_{old}) + ((1 - \lambda) \times msg_{new})$$

**Example Problem:**
* **Computer Vision:** Finding representative exemplar images (like a canonical picture of a cat) from a massive, unlabeled dataset of thousands of diverse images.

In [20]:
from sklearn.cluster import AffinityPropagation
@interact(
    damping=FloatSlider(min=0.5, max=0.99, step=0.05, value=0.5, description='Damping'),
    preference=FloatSlider(min=-200, max=0, step=10, value=-50, description='Preference')
)
def plot_affinity_propagation(damping, preference):
    # Fit Affinity Propagation
    # Preference dictates how many clusters form. Lower (more negative) = fewer clusters.
    aff_prop = AffinityPropagation(damping=damping, preference=preference, random_state=42)
    y_aff = aff_prop.fit_predict(X_blobs)
    
    cluster_centers_indices = aff_prop.cluster_centers_indices_
    n_clusters_ = len(cluster_centers_indices)
    
    plt.figure(figsize=(8, 5))
    
    # Plot data points
    plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_aff, s=40, cmap='plasma', edgecolor='k', alpha=0.7)
    
    # Plot the "Exemplars" (Centers)
    if n_clusters_ > 0:
        centers = aff_prop.cluster_centers_
        plt.scatter(centers[:, 0], centers[:, 1], c='red', s=250, marker='*', edgecolor='white', label='Exemplars')
        
        # Draw lines from exemplars to their cluster members
        for k in range(n_clusters_):
            class_members = y_aff == k
            cluster_center = X_blobs[cluster_centers_indices[k]]
            for x in X_blobs[class_members]:
                plt.plot([cluster_center[0], x[0]], [cluster_center[1], x[1]], color='gray', alpha=0.2, linewidth=0.5)

    plt.title(f"Affinity Propagation (Estimated Clusters: {n_clusters_})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    if n_clusters_ > 0:
        plt.legend()
    plt.show()

interactive(children=(FloatSlider(value=0.5, description='Damping', max=0.99, min=0.5, step=0.05), FloatSlider…

### 7. STING (Statistical Information Grid)

STING is a **Grid-based** clustering algorithm. Instead of calculating distances between thousands of individual data points (which is extremely slow for massive databases), STING divides the spatial area into rectangular cells using a hierarchical grid structure.



It computes statistical parameters (count, mean, variance, min, max) for each cell. Clustering operations are then performed strictly on the *cells* rather than the data points. Once the grid is built, query processing is completely independent of the number of data points, making it incredibly fast ($O(K)$ where $K$ is the number of grid cells).

**Mathematical Foundation (Hierarchical Stats):**
The spatial area is divided into multiple levels of resolution. A cell at a high level (parent) is partitioned into smaller cells at the next lower level (children).

The statistical parameters of a parent cell can be easily computed from the parameters of its child cells. For example:
* **Count:** $N = \sum_{i} n_i$ (Sum of child counts)
* **Mean:** $m = \frac{1}{N} \sum_{i} n_i m_i$ (Weighted average of child means)
* **Min / Max:** $min = \min(min_i)$, $max = \max(max_i)$

**Example Problem:**
* **Geographic Information Systems (GIS):** Quickly clustering millions of GPS coordinate pings from ride-sharing cars by dividing a city map into a grid and analyzing traffic density per city block.

*(Note: Standard scikit-learn does not implement STING. Below is a custom visualization of how a grid-based algorithm maps spatial density.)*

In [ ]:
import matplotlib.patches as patches

@interact(grid_size=IntSlider(min=3, max=15, step=1, value=6, description='Grid Bins'))
def plot_sting_concept(grid_size):
    plt.figure(figsize=(8, 5))
    
    # Create 2D histogram to simulate Grid Cells
    H, xedges, yedges = np.histogram2d(X_moons[:,0], X_moons[:,1], bins=grid_size)
    
    # Plot original data
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c='gray', s=10, alpha=0.5, label='Raw Data')
    
    ax = plt.gca()
    max_count = H.max()
    
    # Draw the STING grid cells
    for i in range(grid_size):
        for j in range(grid_size):
            count = H[i, j]
            if count > 0:
                # Color intensity based on point density in the cell
                alpha_val = min(1.0, count / max_count)
                rect = patches.Rectangle((xedges[i], yedges[j]), 
                                         xedges[i+1]-xedges[i], yedges[j+1]-yedges[j], 
                                         linewidth=1, edgecolor='red', facecolor='blue', alpha=alpha_val * 0.7)
                ax.add_patch(rect)
                
                # Annotate the "Statistic" (Count) of the cell
                plt.text(xedges[i] + (xedges[i+1]-xedges[i])/2, yedges[j] + (yedges[j+1]-yedges[j])/2, 
                         int(count), color='white', ha='center', va='center', fontweight='bold', fontsize=8)

    plt.title(f"STING Concept: Spatial Grid Statistics (Bins={grid_size}x{grid_size})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.show()

interactive(children=(IntSlider(value=6, description='Grid Bins', max=15, min=3), Output()), _dom_classes=('wi…

### 8. CLIQUE (Clustering In QUEst)

**What is CLIQUE?**
CLIQUE beautifully combines **Grid-based** and **Density-based** clustering, specifically designed for finding clusters in high-dimensional data. 

In high dimensions, data becomes extremely sparse (The Curse of Dimensionality), and standard distance metrics fail. CLIQUE solves this by automatically finding sub-spaces (combinations of specific features) where clusters actually exist.

**Mathematical Foundation (The Apriori Principle):**
1. **Grid Partitioning:** Like STING, it partitions the $d$-dimensional data space into non-overlapping rectangular units.
2. **Density Threshold ($\tau$):** A unit is considered **dense** if the fraction of total data points contained within it exceeds a user-defined parameter $\tau$.
   $$\text{Density}(u) = \frac{\text{Count}(x \in u)}{N} \ge \tau$$
3. **Subspace Monotonicity (Apriori Property):** If a collection of points forms a dense cluster in a $k$-dimensional space, then they *must* also form a dense cluster in all $(k-1)$-dimensional projections of that space. 
4. **Cluster Generation:** CLIQUE finds dense 1D units, combines them to find dense 2D units, and so on. Finally, it merges adjacent dense units in the identical subspace to form complex, arbitrary-shaped clusters.

**Example Problem:**
* **Genomics:** Finding groups of patients who share similar expressions across a *subset* of 50 genes out of a massive 10,000-gene sequence profile.

*(Note: Standard scikit-learn does not implement CLIQUE. Below is a custom visualization of the CLIQUE grid-density cluster merging phase.)*

In [23]:
@interact(
    bins=IntSlider(min=5, max=100, step=2, value=15, description='Grid Intervals'),
    density_threshold=FloatSlider(min=0.01, max=0.1, step=0.01, value=0.03, description='Threshold (τ)')
)
def plot_clique_concept(bins, density_threshold):
    plt.figure(figsize=(8, 5))
    
    # Calculate density threshold count
    N = len(X_moons)
    min_points_per_cell = N * density_threshold
    
    # Partition into grid
    H, xedges, yedges = np.histogram2d(X_moons[:,0], X_moons[:,1], bins=bins)
    
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c='black', s=15, alpha=0.4, label='Data')
    
    ax = plt.gca()
    dense_cells_found = 0
    
    # Identify "Dense Units" and merge them (visually)
    for i in range(bins):
        for j in range(bins):
            # CLIQUE checks if cell density > tau
            if H[i, j] >= min_points_per_cell:
                dense_cells_found += 1
                rect = patches.Rectangle((xedges[i], yedges[j]), 
                                         xedges[i+1]-xedges[i], yedges[j+1]-yedges[j], 
                                         linewidth=1.5, edgecolor='black', facecolor='lime', alpha=0.6)
                ax.add_patch(rect)

    plt.title(f"CLIQUE Concept: Merging Dense Units (Threshold > {int(min_points_per_cell)} points)", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    
    # Custom legend for the dense patches
    import matplotlib.lines as mlines
    dense_patch = mlines.Line2D([], [], color='lime', marker='s', linestyle='None', 
                                markersize=10, alpha=0.6, label=f'Dense Units ({dense_cells_found})')
    handles, labels = ax.get_legend_handles_labels()
    handles.append(dense_patch)
    plt.legend(handles=handles, loc='upper right')
    
    plt.show()

interactive(children=(IntSlider(value=15, description='Grid Intervals', min=5, step=2), FloatSlider(value=0.03…